# External validation on BDD100K - YOLOv11s trained on DAWN seed 42


Purpose:
Evaluates the YOLOv11s model trained on the DAWN dataset using the independent BDD100K dataset (seed 42) to assess external generalisation performance.

Contents:

Loading the trained YOLOv11s model (seed 42).
Running inference on the BDD100K test set.
Computing detection performance metrics.
Exporting evaluation results for further analysis.

# Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


# Install Ultralytics

In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.6 MB/s eta 0:00:00


# Imports

In [ ]:
from pathlib import Path
from datetime import datetime

import json
import yaml
import numpy as np
import pandas as pd
import torch
import ultralytics

from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Check environment

In [ ]:
print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: GPU is not available.")

Ultralytics version: 8.4.100
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


# Main Config

In [ ]:
# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================

MODEL_NAME = "YOLOv11s"

# Dataset utilisé pour entraîner le modèle
TRAINING_DATASET = "DAWN"

# Dataset externe utilisé pour le testing
TESTING_DATASET = "BDD100K"

SEED = 42

# ------------------------------------------------------------
# Path to the trained YOLO model
# ------------------------------------------------------------

MODEL_PATH = Path(
    "/content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/weights/best.pt"
)

# ------------------------------------------------------------
# BDD100K YOLO dataset
# ------------------------------------------------------------

BDD100K_ROOT = Path(
    "/content/drive/MyDrive/Dissertation/Datasets/processed/BDD100K"
)

BDD100K_YOLO_ROOT = BDD100K_ROOT / "yolo"
BDD100K_YAML = BDD100K_YOLO_ROOT / "dataset.yaml"

BDD100K_VAL_IMAGES = BDD100K_YOLO_ROOT / "images" / "val"
BDD100K_VAL_LABELS = BDD100K_YOLO_ROOT / "labels" / "val"


# Raw model outputs
RUNS_ROOT = Path(
    "/content/drive/MyDrive/Dissertation/Runs/yolo/dawn_bddk_seed42"
)

EXPERIMENT_NAME = f"{MODEL_NAME}_{TRAINING_DATASET}_to_{TESTING_DATASET}_seed_{SEED}"
RUN_OUTPUT_DIR = RUNS_ROOT / EXPERIMENT_NAME

# Clean metrics for final analysis
RESULTS_ROOT = Path(
    "/content/drive/MyDrive/Dissertation/Results/bddk_crossdataset"
)

METRICS_OUTPUT_DIR = (
    RESULTS_ROOT / MODEL_NAME /
    f"{TRAINING_DATASET}_to_{TESTING_DATASET}" /
    f"seed_{SEED}"
)

RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Experiment:", EXPERIMENT_NAME)
print("Raw run output:", RUN_OUTPUT_DIR)
print("Clean metrics output:", METRICS_OUTPUT_DIR)


Experiment: YOLOv11s_DAWN_to_BDD100K_seed_42
Raw run output: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_bddk_seed42/YOLOv11s_DAWN_to_BDD100K_seed_42
Clean metrics output: /content/drive/MyDrive/Dissertation/Results/bddk_crossdataset/YOLOv11s/DAWN_to_BDD100K/seed_42


### Check paths

In [ ]:
path_checks = {
    "Model best.pt": MODEL_PATH,
    "BDD100K root": BDD100K_ROOT,
    "BDD100K YOLO root": BDD100K_YOLO_ROOT,
    "BDD100K dataset.yaml": BDD100K_YAML,
    "BDD100K validation images": BDD100K_VAL_IMAGES,
    "BDD100K validation labels": BDD100K_VAL_LABELS,
}

all_paths_valid = True
for name, path in path_checks.items():
    exists = path.exists()
    print(f"{name}: {exists}")
    print(f"  {path}\n")
    if not exists:
        all_paths_valid = False

if not all_paths_valid:
    raise FileNotFoundError("One or more required paths do not exist.")

print("All required paths exist.")


Model best.pt: True
  /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/weights/best.pt

BDD100K root: True
  /content/drive/MyDrive/Dissertation/Datasets/processed/BDD100K

BDD100K YOLO root: True
  /content/drive/MyDrive/Dissertation/Datasets/processed/BDD100K/yolo

BDD100K dataset.yaml: True
  /content/drive/MyDrive/Dissertation/Datasets/processed/BDD100K/yolo/dataset.yaml

BDD100K validation images: True
  /content/drive/MyDrive/Dissertation/Datasets/processed/BDD100K/yolo/images/val

BDD100K validation labels: True
  /content/drive/MyDrive/Dissertation/Datasets/processed/BDD100K/yolo/labels/val

All required paths exist.


# Count validation images & labels

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

validation_images = sorted(
    p for p in BDD100K_VAL_IMAGES.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)
validation_labels = sorted(BDD100K_VAL_LABELS.glob("*.txt"))

print("Validation images:", len(validation_images))
print("Validation label files:", len(validation_labels))


Validation images: 1504
Validation label files: 1504


### Verify image-label correspondance

In [ ]:
image_stems = {p.stem for p in validation_images}
label_stems = {p.stem for p in validation_labels}

images_without_label_file = sorted(image_stems - label_stems)
labels_without_image = sorted(label_stems - image_stems)

print("Images without label file:", len(images_without_label_file))
print("Labels without corresponding image:", len(labels_without_image))

if images_without_label_file:
    print("\nFirst images without labels:", images_without_label_file[:10])
if labels_without_image:
    print("\nFirst labels without images:", labels_without_image[:10])


Images without label file: 0
Labels without corresponding image: 0


# Read & verify dataset.yaml

In [ ]:
with open(BDD100K_YAML, "r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

print(json.dumps(dataset_config, indent=4))

EXPECTED_CLASSES = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "bus",
    5: "truck",
}

yaml_names = dataset_config.get("names")
if isinstance(yaml_names, list):
    yaml_classes = {i: name for i, name in enumerate(yaml_names)}
else:
    yaml_classes = {int(i): name for i, name in yaml_names.items()}

if yaml_classes != EXPECTED_CLASSES:
    raise ValueError("BDD100K class names/order do not match the expected six classes.")

print("BDD100K class configuration is correct.")


{
    "path": "/content/drive/MyDrive/Dissertation/Datasets/processed/BDD100K/yolo",
    "train": "images/train",
    "val": "images/val",
    "names": {
        "0": "person",
        "1": "bicycle",
        "2": "car",
        "3": "motorcycle",
        "4": "bus",
        "5": "truck"
    }
}
BDD100K class configuration is correct.


# Validate YOLO labels and count objects

In [ ]:
invalid_annotations = []
class_counts = {class_id: 0 for class_id in EXPECTED_CLASSES}

for label_path in validation_labels:
    with open(label_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    for line_number, line in enumerate(lines, start=1):
        values = line.strip().split()
        if not values:
            continue
        if len(values) != 5:
            invalid_annotations.append({"file": str(label_path), "line": line_number, "reason": "Expected 5 values"})
            continue
        try:
            class_id = int(float(values[0]))
            x_center, y_center, width, height = map(float, values[1:])
        except ValueError:
            invalid_annotations.append({"file": str(label_path), "line": line_number, "reason": "Non-numeric value"})
            continue
        if class_id not in EXPECTED_CLASSES:
            invalid_annotations.append({"file": str(label_path), "line": line_number, "reason": f"Invalid class ID {class_id}"})
            continue
        if not all(0 <= v <= 1 for v in [x_center, y_center, width, height]):
            invalid_annotations.append({"file": str(label_path), "line": line_number, "reason": "Coordinate outside [0,1]"})
            continue
        if width <= 0 or height <= 0:
            invalid_annotations.append({"file": str(label_path), "line": line_number, "reason": "Invalid box size"})
            continue
        class_counts[class_id] += 1

print("Invalid annotations:", len(invalid_annotations))
if invalid_annotations:
    display(pd.DataFrame(invalid_annotations).head(20))
else:
    print("All checked YOLO annotations are valid.")


Invalid annotations: 0
All checked YOLO annotations are valid.


# Save class distribution

In [ ]:
class_distribution_df = pd.DataFrame([
    {
        "class_id": class_id,
        "class_name": EXPECTED_CLASSES[class_id],
        "object_count": class_counts[class_id],
    }
    for class_id in EXPECTED_CLASSES
])

display(class_distribution_df)
class_distribution_df.to_csv(
    METRICS_OUTPUT_DIR / "bdd100k_validation_class_distribution.csv",
    index=False,
)


,class_id,class_name,object_count
0,0,person,1963
1,1,bicycle,138
2,2,car,14573
3,3,motorcycle,40
4,4,bus,265
5,5,truck,754


# Load and verify the trained model

In [ ]:
model = YOLO(str(MODEL_PATH))
print("Model loaded successfully.")
print("Model classes:", model.names)

model_classes = {int(i): name for i, name in model.names.items()}
if model_classes != EXPECTED_CLASSES:
    raise ValueError(
        "The model classes do not match the six BDD100K classes. "
        "Do not continue with evaluation."
    )

print("Model classes match the BDD100K classes.")


Model loaded successfully.
Model classes: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'bus', 5: 'truck'}
Model classes match the BDD100K classes.


# Remove old Ultralytics cache files

In [ ]:
cache_files = list(BDD100K_YOLO_ROOT.rglob("*.cache"))
print("Cache files found:", len(cache_files))
for cache_file in cache_files:
    print("Deleting:", cache_file)
    cache_file.unlink()
print("Cache cleanup completed.")


Cache files found: 1
Deleting: /content/drive/MyDrive/Dissertation/Datasets/processed/BDD100K/yolo/labels/val.cache
Cache cleanup completed.


# Run BDD100K evaluation

In [ ]:
BATCH_SIZE = 16

evaluation_results = model.val(
    data=str(BDD100K_YAML),
    split="val",
    imgsz=640,
    batch=BATCH_SIZE,
    device=0,
    workers=2,
    conf=0.001,
    iou=0.60,
    plots=True,
    save_json=True,
    project=str(RUNS_ROOT),
    name=EXPERIMENT_NAME,
    exist_ok=True,
    verbose=True,
)


Ultralytics 8.4.100 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.8±0.2 ms, read: 0.2±0.0 MB/s, size: 62.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/BDD100K/yolo/labels/val... 1504 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1504/1504 55.2it/s 27.3s
val: New cache created: /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/BDD100K/yolo/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 8.9it/s 10.6s
                   all       1504      17733      0.354      0.146   

# Extract global metrics and timing

In [ ]:
precision = float(evaluation_results.box.mp)
recall = float(evaluation_results.box.mr)
f1_score = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

speed = evaluation_results.speed
preprocess_ms = float(speed.get("preprocess", 0.0))
inference_ms = float(speed.get("inference", 0.0))
loss_ms = float(speed.get("loss", 0.0))
postprocess_ms = float(speed.get("postprocess", 0.0))
total_pipeline_ms = preprocess_ms + inference_ms + postprocess_ms

complete_global_metrics = {
    "model": MODEL_NAME,
    "training_dataset": TRAINING_DATASET,
    "testing_dataset": TESTING_DATASET,
    "evaluation_subset": "BDD100K validation",
    "seed": SEED,
    "number_of_images": len(validation_images),
    "mAP50_95": float(evaluation_results.box.map),
    "mAP50": float(evaluation_results.box.map50),
    "mAP75": float(evaluation_results.box.map75),
    "precision": precision,
    "recall": recall,
    "f1_score": f1_score,
    "preprocess_ms_per_image": preprocess_ms,
    "inference_ms_per_image": inference_ms,
    "loss_ms_per_image": loss_ms,
    "postprocess_ms_per_image": postprocess_ms,
    "total_pipeline_ms_per_image": total_pipeline_ms,
    "fps_inference_only": 1000.0 / inference_ms if inference_ms > 0 else None,
    "fps_total_pipeline": 1000.0 / total_pipeline_ms if total_pipeline_ms > 0 else None,
    "image_size": 640,
    "batch_size": BATCH_SIZE,
    "confidence_threshold": 0.001,
    "nms_iou_threshold": 0.60,
    "model_path": str(MODEL_PATH),
    "dataset_yaml": str(BDD100K_YAML),
    "raw_run_directory": str(RUN_OUTPUT_DIR),
    "clean_metrics_directory": str(METRICS_OUTPUT_DIR),
    "evaluation_date": datetime.now().isoformat(),
}

complete_global_metrics_df = pd.DataFrame([complete_global_metrics])
display(complete_global_metrics_df)


,model,training_dataset,testing_dataset,evaluation_subset,seed,number_of_images,mAP50_95,mAP50,mAP75,precision,...,fps_total_pipeline,image_size,batch_size,confidence_threshold,nms_iou_threshold,model_path,dataset_yaml,raw_run_directory,clean_metrics_directory,evaluation_date
0,YOLOv11s,DAWN,BDD100K,BDD100K validation,42,1504,0.064378,0.119794,0.060927,0.354411,...,421.36586,640,16,0.001,0.6,/content/drive/MyDrive/Dissertation/Runs/yolo/...,/content/drive/MyDrive/Dissertation/Datasets/p...,/content/drive/MyDrive/Dissertation/Runs/yolo/...,/content/drive/MyDrive/Dissertation/Results/bd...,2026-07-18T10:54:47.165704


# Extract per-class metrics

In [ ]:
per_class_maps = np.asarray(evaluation_results.box.maps, dtype=float)
per_class_precision = np.asarray(evaluation_results.box.p, dtype=float)
per_class_recall = np.asarray(evaluation_results.box.r, dtype=float)
per_class_f1 = np.asarray(evaluation_results.box.f1, dtype=float)

per_class_rows = []
for class_id, class_name in EXPECTED_CLASSES.items():
    per_class_rows.append({
        "model": MODEL_NAME,
        "training_dataset": TRAINING_DATASET,
        "testing_dataset": TESTING_DATASET,
        "seed": SEED,
        "class_id": class_id,
        "class_name": class_name,
        "ground_truth_object_count": class_counts.get(class_id, 0),
        "mAP50_95": float(per_class_maps[class_id]) if class_id < len(per_class_maps) else np.nan,
        "precision": float(per_class_precision[class_id]) if class_id < len(per_class_precision) else np.nan,
        "recall": float(per_class_recall[class_id]) if class_id < len(per_class_recall) else np.nan,
        "f1_score": float(per_class_f1[class_id]) if class_id < len(per_class_f1) else np.nan,
    })

per_class_metrics_df = pd.DataFrame(per_class_rows)
display(per_class_metrics_df)


,model,training_dataset,testing_dataset,seed,class_id,class_name,ground_truth_object_count,mAP50_95,precision,recall,f1_score
0,YOLOv11s,DAWN,BDD100K,42,0,person,1963,0.074498,0.561676,0.175751,0.267729
1,YOLOv11s,DAWN,BDD100K,42,1,bicycle,138,0.004325,0.581550,0.014493,0.028281
2,YOLOv11s,DAWN,BDD100K,42,2,car,14573,0.230387,0.728685,0.359089,0.481097
3,YOLOv11s,DAWN,BDD100K,42,3,motorcycle,40,0.000152,0.000000,0.000000,0.000000
4,YOLOv11s,DAWN,BDD100K,42,4,bus,265,0.034149,0.133654,0.120755,0.126877
5,YOLOv11s,DAWN,BDD100K,42,5,truck,754,0.042757,0.120900,0.205570,0.152256


# Save clean metrics

In [ ]:
GLOBAL_CSV_PATH = METRICS_OUTPUT_DIR / "global_metrics.csv"
GLOBAL_JSON_PATH = METRICS_OUTPUT_DIR / "global_metrics.json"
PER_CLASS_CSV_PATH = METRICS_OUTPUT_DIR / "per_class_metrics.csv"
PER_CLASS_JSON_PATH = METRICS_OUTPUT_DIR / "per_class_metrics.json"

complete_global_metrics_df.to_csv(GLOBAL_CSV_PATH, index=False)
per_class_metrics_df.to_csv(PER_CLASS_CSV_PATH, index=False)

with open(GLOBAL_JSON_PATH, "w", encoding="utf-8") as file:
    json.dump(complete_global_metrics, file, indent=4, ensure_ascii=False)

with open(PER_CLASS_JSON_PATH, "w", encoding="utf-8") as file:
    json.dump(per_class_metrics_df.to_dict(orient="records"), file, indent=4, ensure_ascii=False)

print("Global metrics:", GLOBAL_CSV_PATH)
print("Per-class metrics:", PER_CLASS_CSV_PATH)


Global metrics: /content/drive/MyDrive/Dissertation/Results/bddk_crossdataset/YOLOv11s/DAWN_to_BDD100K/seed_42/global_metrics.csv
Per-class metrics: /content/drive/MyDrive/Dissertation/Results/bddk_crossdataset/YOLOv11s/DAWN_to_BDD100K/seed_42/per_class_metrics.csv


# Save experiment configuration

In [ ]:
experiment_configuration = {
    "experiment_name": EXPERIMENT_NAME,
    "model": MODEL_NAME,
    "training_dataset": TRAINING_DATASET,
    "testing_dataset": TESTING_DATASET,
    "seed": SEED,
    "model_path": str(MODEL_PATH),
    "dataset_yaml": str(BDD100K_YAML),
    "evaluation_split": "val",
    "evaluation_role": "external cross-dataset testing",
    "number_of_images": len(validation_images),
    "number_of_label_files": len(validation_labels),
    "image_size": 640,
    "batch_size": BATCH_SIZE,
    "device": 0,
    "workers": 2,
    "confidence_threshold": 0.001,
    "nms_iou_threshold": 0.60,
    "classes": EXPECTED_CLASSES,
    "raw_run_directory": str(RUN_OUTPUT_DIR),
    "clean_metrics_directory": str(METRICS_OUTPUT_DIR),
    "evaluation_date": datetime.now().isoformat(),
}

CONFIG_PATH = METRICS_OUTPUT_DIR / "evaluation_configuration.json"
with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(experiment_configuration, file, indent=4, ensure_ascii=False)

print("Configuration saved:", CONFIG_PATH)


Configuration saved: /content/drive/MyDrive/Dissertation/Results/bddk_crossdataset/YOLOv11s/DAWN_to_BDD100K/seed_42/evaluation_configuration.json


# Save random prediction examples in Runs

In [ ]:
rng = np.random.default_rng(SEED)
number_of_samples = min(10, len(validation_images))
selected_indices = rng.choice(len(validation_images), size=number_of_samples, replace=False)
random_sample_images = [validation_images[index] for index in selected_indices]

sample_predictions = model.predict(
    source=[str(path) for path in random_sample_images],
    imgsz=640,
    conf=0.25,
    iou=0.60,
    device=0,
    save=True,
    project=str(RUN_OUTPUT_DIR),
    name="sample_predictions",
    exist_ok=True,
    verbose=False,
)

print("Sample predictions saved to:", RUN_OUTPUT_DIR / "sample_predictions")


Results saved to /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Runs/yolo/dawn_bddk_seed42/YOLOv11s_DAWN_to_BDD100K_seed_42/sample_predictions
Sample predictions saved to: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_bddk_seed42/YOLOv11s_DAWN_to_BDD100K_seed_42/sample_predictions
